# Contest workflow and reproducibility

**M0 · Toolbox & Submission Craft** — [AI Olympiad Curriculum](https://ai-olympiad-curriculum.netlify.app)

~30 min

## Read the task like a lawyer

- **Metric first**: F1 vs accuracy vs log-loss change which model wins. Optimize the leaderboard metric, not a convenient one.
- **Submission format**: column names, row order, id field, probability vs hard class. One wrong header and a perfect model scores zero.
- **Limits**: wall-clock time, RAM, no internet, allowed libraries, whether the jury re-runs your notebook.
- **Forbidden**: external data, pretrained weights, hand-labelling test rows, multiple accounts.

> **Note.** Print the sample submission and make yours `diff`-clean against it: same columns, same order, same index. Check `submission.shape` and `submission.columns` before uploading.



## Baseline first, then one change at a time

- Make a trivial valid submission first (constant prediction or the given baseline). It exercises the whole pipeline — if the plumbing is broken, you find out in 10 minutes, not on the last day.
- Build a **validation** set — a held-out slice of train that stands in for the hidden test — and mirror its split logic (by user, by time). Trust it more than the public leaderboard.
- Change one thing at a time and log the score of every experiment — the next step makes that a habit.



## Keep an experiment log

After run 10 you will not remember what run 3 did. A log is one row per experiment: what changed, validation score, seed, runtime. A list of dicts is enough — the discipline matters more than the tool.



In [ ]:
import pandas as pd
log = []
def record(name, val_score, **cfg):       # **cfg collects extra keyword arguments into a dict
    log.append({"name": name, "val": val_score, **cfg})   # **cfg spreads them into the new dict

record("baseline",   0.812, model="logreg")
record("more trees", 0.851, model="rf", n=500)
pd.DataFrame(log).sort_values("val", ascending=False)

The log turns "I think deeper trees helped" into a lookup — and it stops you re-tuning a knob you already swept.



## Reproducibility



In [ ]:
import random, numpy as np, torch
def seed_all(s=42):
    # Python, NumPy and PyTorch each keep a separate random generator —
    # seeding one does not seed the others, so all three calls are needed
    random.seed(s); np.random.seed(s)
    torch.manual_seed(s); torch.cuda.manual_seed_all(s)
seed_all()

Juries re-run notebooks. Fix seeds, note your `sklearn`/`torch` versions, and replace hard-coded local paths with relative ones (`Path("data") / "train.csv"`, not `C:\Users\you\...`). Time one full top-to-bottom run against the limit.

GPUs are not bit-deterministic — expect small numerical drift across hardware and leave margin in your score. If two runs with the same seed disagree wildly, you have hidden randomness: an unseeded shuffle, `set` iteration order, or `drop_last`-style nondeterminism in a loader.



### ✏️ Your validation score improves but the leaderboard score drops. Most likely cause?

**A.** Validation doesn't match the test distribution, or it leaks
**B.** The leaderboard metric weights errors differently than the local one
**C.** The submission file was uploaded with rows in the wrong order
**D.** The GPU gives slightly different numerical results on the test set

<details><summary>Answer</summary>

**A.** Validation doesn't match the test distribution, or it leaks

Validation must be built the way the test was built (grouped by user, split by time, etc.).

</details>

### ✏️ A notebook gives a great score but fails when re-run fresh top-to-bottom. Cause?

**A.** The metric used a different random seed by default
**B.** Restarting cleared the model's weight cache
**C.** Stale state from cells run out of order earlier
**D.** The GPU produced luckier randomness the first time

<details><summary>Answer</summary>

**C.** Stale state from cells run out of order earlier

Notebooks keep variables from deleted or edited cells; only a fresh-kernel run proves the result is real.

</details>

---

**Go deeper**
- [IOAI official task repos](https://github.com/IOAI-official)